# Differentially Methylated Positions in Fish Mitochondrial Genome Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.3383-hnem/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata._name}: {metadata._description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We'll extract a list of all record sets, fields, and columns using their `@id` values. This is important for reproducibility and for referencing the data with Croissant-compliant tools.

In [ ]:
record_sets_info = []
for rs in dataset.metadata._record_sets:
    print(f"RecordSet: @id={rs['@id']}, name='{rs.get('name', '-')}'")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print('  Fields:')
    for field in fields:
        print(f"   - @id={field['@id']}, name='{field.get('name', '-')}', dataType={field.get('dataType', '-')} ")
        # You can add columns if available as well
    record_sets_info.append(rs['@id'])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, let's enumerate the available record set @ids
from IPython.display import display

record_sets = [rs['@id'] for rs in dataset.metadata._record_sets]
dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:  # Only create DataFrame if records exist
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"Loaded {len(df)} records for record_set: {record_set}")
            print("Fields:", df.columns.tolist())
            display(df.head())
    except Exception as e:
        print(f"Could not load record_set '{record_set}': {e}")

# For the next cells, select the main table to analyze: choose a record_set that contains the main quantitative data (the one about positions, gene, methylation, etc.)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Analyzing record_set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All field and record set/entity references below are made **by their `@id`** for clarity and reproducibility. Adjust the parameters as required based on your data.

In [ ]:
import numpy as np
import warnings

# For demo purposes, we need to pick the numeric and group fields using their IDs
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Available fields: {df.columns.tolist()}")
    # Try to find likely numeric and group fields (you may want to edit these for your use-case or look them up above!)
    # Assume field IDs are used as DataFrame column names (as per mlcroissant)
    candidate_numeric = None
    candidate_group = None
    for col in df.columns:
        # Try to find a numeric field
        if candidate_numeric is None and pd.api.types.is_numeric_dtype(df[col]):
            candidate_numeric = col
        # Try to find a grouping/categorical field
        if candidate_group is None and not pd.api.types.is_numeric_dtype(df[col]):
            candidate_group = col

    if not candidate_numeric:
        warnings.warn("No numeric column found; please set candidate_numeric manually.")
    else:
        numeric_field_id = candidate_numeric

        # Set a threshold using the numeric field's mean as an example
        threshold = df[numeric_field_id].mean()
        print(f"Filtering rows where {numeric_field_id} > {threshold:.2f}")

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} found.")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized column '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by the candidate group field
        group_field_id = candidate_group
        if group_field_id is not None and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (showing means):")
            display(grouped_df)
else:
    print("No main quantitative record set found to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    if candidate_numeric:
        plt.figure(figsize=(7,4))
        sns.histplot(df[candidate_numeric], kde=True)
        plt.title(f"Distribution of numeric field ({candidate_numeric})")
        plt.show()

    if candidate_numeric and candidate_group and candidate_group in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[candidate_group], y=df[candidate_numeric])
        plt.title(f"{candidate_numeric} by {candidate_group}")
        plt.xlabel(candidate_group)
        plt.ylabel(candidate_numeric)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the <b>Differentially Methylated Positions in Fish Mitochondrial Genome</b> dataset from a Croissant schema.
- We queried record sets, fields, and extracted data by their `@id`s, ensuring reproducibility.
- We performed basic data cleaning, filtering, normalization, and grouping on quantitative records.
- Simple visualizations (distribution and box plots) revealed data structure and possible group differences.

Further exploration could involve:
- Advanced filtering and statistical tests on methylation differences.
- Correlational analyses of methylation vs. gene or group categories.
- Integration with biological/functional databases for annotated insights.